In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [ ]:
import os #Dùng để làm việc với hệ điều hành (operating system)
os.listdir('/content')

['.config', 'email_cleaned.xlsx', 'sample_data']

In [ ]:
# Đọc file excel
df = pd.read_csv('emails.csv')

# Hiển thị 5 dòng đầu
df.head()

,Unnamed: 0,text,spam,Cleaned
0,0,Subject: naturally irresistible your corporate...,1,subject natur irresist corpor ident lt realli ...
1,1,Subject: the stock trading gunslinger fanny i...,1,subject stock trade gunsling fanni merril muzo...
2,2,Subject: unbelievable new homes made easy im ...,1,subject unbeliev new home made easi im want sh...
3,3,Subject: 4 color printing special request add...,1,subject color print special request addit info...
4,4,"Subject: do not have money , get software cds ...",1,subject money get softwar cd softwar compat gr...


In [ ]:
document = []
label = []

# Lấy nội dung email đã xử lý
for e in df["Cleaned"]:
    document.append(e)

# Lấy nhãn spam/ham
for l in df["spam"]:
    label.append(l)

print("Total emails:", len(document))

Total emails: 5728


In [ ]:
# Chia tập train / test
document_train, document_test, label_train, label_test = train_test_split(
    document,
    label,
    test_size=0.2,       # 20% test
    stratify=label,      # Giữ tỷ lệ spam/ham
    random_state=42
)

print("Train size:", len(document_train))
print("Test size:", len(document_test))

Train size: 4582
Test size: 1146


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Tạo vectorizer
vectorizer = CountVectorizer(
    tokenizer=lambda x: x.split(),   # Giống doc.split()
    lowercase=False,                 # Giữ nguyên lowercase và uppercase
    token_pattern=None,              # Tắt regex mặc định vì regex mặc định sẽ chỉ lấy từ >= 2 ký tự ("I am GPT" -> ["am", "GPT"])
    binary=True                      # Chỉ cần 0/1 (word xuất hiện hay không)
)

# Train vocabulary
X_train = vectorizer.fit_transform(document_train) #fit() học từ dữ liệu + transform() biến dữ liệu thành vector số

# Test vocabulary
X_test = vectorizer.transform(document_test)
# X đang là Sparse matrix (Chỉ lưu các giá trị khác 0)
# (0, 1)  1
# (0, 3)  1
# (1, 2)  1

# Chuyển sang numpy array để dễ xử lý
vectors_train = X_train.toarray()
vectors_test = X_test.toarray()

# Vocabulary (danh sách từ)
set_words = vectorizer.get_feature_names_out() # get_feature_names_out(): hàm lấy toàn bộ danh sách từ mà CountVectorizer đã tạo.

print("Số email train:", vectors_train.shape[0])
print("Số từ train:", vectors_train.shape[1])

print("Số email test:", vectors_test.shape[0])
print("Số từ test:", vectors_test.shape[1])

Số email train: 4582
Số từ train: 23074
Số email test: 1146
Số từ test: 23074


In [ ]:
def smoothing(a, b):
    return float((a + 1) / (b + 1))

In [ ]:
spam = 0
non_spam = 0
for i in label_train:
    if i == 1:
        spam += 1
    else:
        non_spam += 1
print("spam:", spam, "/ non_spam:", non_spam)

spam_coef = smoothing(spam, (spam+non_spam)) #Hệ số xác suất của spam
non_spam_coef = smoothing(non_spam, (spam+non_spam)) #Hệ số xác suất của non_spam


spam: 1094 / non_spam: 3488


In [ ]:
bayes_matrix = np.zeros((len(set_words), 4)) #app/spam, app/nonspam, nonapp/spam, nonapp/nonspam

In [ ]:
for i in range(len(set_words)): # Duyệt qua từng từ trong bag_of_words
    app_spam = 0
    app_nonspam = 0
    nonapp_spam = 0
    nonapp_nonspam = 0
    for j, v in enumerate(vectors_train): # Duyệt qua từng email: j là index, v là vector email của email[j]
        if v[i] == 1:
            if label_train[j] == 1:
                app_spam += 1
            else:
                app_nonspam += 1
        else:
            if label_train[j] == 1:
                nonapp_spam += 1
            else:
                nonapp_nonspam += 1

    bayes_matrix[i][0] = smoothing(app_spam, spam)
    bayes_matrix[i][1] = smoothing(app_nonspam, non_spam)
    bayes_matrix[i][2] = smoothing(nonapp_spam, spam)
    bayes_matrix[i][3] = smoothing(nonapp_nonspam, non_spam)


###Phân loại một email mới

In [ ]:
def predict(mail):

    # Tạo vector giống code ban đầu
    vector = np.zeros(len(set_words))
    words = set(mail.split())

    for i, word in enumerate(set_words):
        if word in words:
            vector[i] = 1

    # Khởi tạo log xác suất ban đầu của mỗi lớp
    log_spam = np.log(spam_coef)
    log_nonspam = np.log(non_spam_coef)

    # Tính Naive Bayes
    for i, v in enumerate(vector):
        if v == 1: # Từ có xuất hiện trong email
            log_spam += np.log(bayes_matrix[i][0])
            log_nonspam += np.log(bayes_matrix[i][1])
        else: # Từ ko xuất hiện trong email
            log_spam += np.log(bayes_matrix[i][2])
            log_nonspam += np.log(bayes_matrix[i][3])

    # So sánh
    if log_spam > log_nonspam:
        return 1
    else:
        return 0

In [ ]:
pred = []

for mail in document_test:
    result = predict(mail)
    pred.append(result)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(label_test, pred)
precision = precision_score(label_test, pred)
recall = recall_score(label_test, pred)
f1 = f1_score(label_test, pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

print("\nConfusion Matrix:")
print(confusion_matrix(label_test, pred))

print("\nModel Evaluation:")

if f1 >= 0.9:
    print("Mô hình rất tốt (excellent model)")
elif f1 >= 0.8:
    print("Mô hình tốt (good model)")
elif f1 >= 0.7:
    print("Mô hình khá (acceptable model)")
elif f1 >= 0.6:
    print("Mô hình yếu (weak model)")
else:
    print("Mô hình kém (poor model)")

Accuracy : 0.9703315881326352
Precision: 0.8921568627450981
Recall   : 0.9963503649635036
F1-score : 0.9413793103448276

Confusion Matrix:
[[839  33]
 [  1 273]]

Model Evaluation:
Mô hình rất tốt (excellent model)
